In [1]:
# Deep Learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader

# Audio processing
import torchaudio
import torchaudio.transforms as T
import librosa

# Pre-trained image models
import timm

# Play the audio in Jupyter notebook
from IPython.display import Audio
import pandas as pd
import os

c:\Users\tomsb\anaconda3\envs\study\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
AUDIO_DIR = "audios/labeled/"

dict_genres = {'positive': 0, 'negative': 1, 'noise': 2}

reverse_map = {v: k for k, v in dict_genres.items()}
print(reverse_map)


{0: 'positive', 1: 'negative', 2: 'noise'}


In [12]:
data = []

for label in dict_genres.keys():
    for folder in os.listdir(AUDIO_DIR + label):
        for file in os.listdir(AUDIO_DIR + label + "/" + folder):
            file_path = AUDIO_DIR + label + "/" + folder + "/" + file
            positive = int(label == "positive")
            negative = int(label == "negative")
            noise = int(label == "noise")
            data.append((file_path, positive,
                        negative, noise))

file_path, positive, negative, noise = zip(*data)
df = pd.DataFrame({"file_path": file_path, "positive": positive, "negative": negative, "noise": noise})
print(df.head(5))


                                 file_path  positive  negative  noise
0  audios/labeled/positive/1001/100100.wav         1         0      0
1  audios/labeled/positive/1002/100200.wav         1         0      0
2  audios/labeled/positive/1003/100300.wav         1         0      0
3  audios/labeled/positive/1003/100301.wav         1         0      0
4  audios/labeled/positive/1003/100302.wav         1         0      0


In [4]:
class AudioDataset(Dataset):
    def __init__(self, 
                df,
                audio_length,
                target_sample_rate=16000):
        self.df = df
        self.file_paths = df['file_path'].values
        self.labels = df[['positive', 'negative', 'noise']].values
        self.target_sample_rate = target_sample_rate
        self.num_samples = target_sample_rate * audio_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):

        # Load audio from file to waveform
        audio, sample_rate = torchaudio.load(self.file_paths[index])

        # Convert to mono
        audio = torch.mean(audio, axis=0)

        # Resample
        if sample_rate != self.target_sample_rate:
            resample = T.Resample(sample_rate, self.target_sample_rate)
            audio = resample(audio)
        
        # Adjust number of samples
        if audio.shape[0] > self.num_samples:
            # Crop
            audio = audio[:self.num_samples]
        elif audio.shape[0] < self.num_samples:
            # Pad
            audio = F.pad(audio, (0, self.num_samples - audio.shape[0]))


        # Add any preprocessing you like here 
        # (e.g., noise removal, etc.)
        
        
        # Add any data augmentations for waveform you like here
        # (e.g., noise injection, shifting time, changing speed and pitch)
        """ 
        wave_transforms = T.PitchShift(sample_rate, 4)
        audio = wave_transforms(audio)
        """
        # Convert to Mel spectrogram
        melspectrogram = T.MelSpectrogram(sample_rate = self.target_sample_rate, 
                                        n_mels = 128, 
                                        n_fft = 2048, 
                                        hop_length = 512)
        melspec = melspectrogram(audio)
        
        # Add any data augmentations for spectrogram you like here
        # (e.g., Mixup, cutmix, time masking, frequency masking)
        """ 
        spec_transforms = T.FrequencyMasking(freq_mask_param=80)
        melspec = spec_transforms(melspec)
        """
        return {"image": torch.stack([melspec]), 
                "label": torch.tensor(self.labels[index]).float()}

In [5]:
class AudioModel(nn.Module):
    def __init__(self,
                 num_classes,
                 model_name='tf_efficientnet_b3_ns',
                 pretrained=True):
        super(AudioModel, self).__init__()

        self.model = timm.create_model(model_name,
                                       pretrained=pretrained,
                                       in_chans=1)
        self.in_features = self.model.classifier.in_features
        self.model.classifier = nn.Sequential(
            nn.Linear(self.in_features, num_classes)
        )

    def forward(self, images):
        logits = self.model(images)
        return logits


In [ ]:
dataset = AudioDataset(df=df, audio_length=)